<div style="border-left: 4px solid #2980b9; padding-left: 15px; margin-bottom: 20px;">
  <h1 style="margin-top: 0;">🛠️🔍 02. Machine Learning: Anomaly & Degradation Monitoring</h1>
  <p style="font-size: 1.1em; color: #888;">Recognizing unusual industrial energy behavior, separating missing data from real observations, and preparing reliable review signals for short-term scheduling.</p>
</div>

### 🎯 Main Goals & Business Value
**Can we spot unusual machine behavior early, understand its context, and make smarter energy decisions without pretending we know when a machine will fail?**

This notebook follows the **15-minute forecasting benchmark** and asks five questions:

1. **Can we trust the measurements?** Missing data are not equipment failures.
2. **What looks unusual?** Compare each interval with similar historical operating conditions.
3. **When did unusual behavior occur?** Show readable charts and review events.
4. **Does forecasting confirm the unusual behavior?** Use the actual forecasts from Notebook 01 *only after the corresponding measurements exist*.
5. **Is the equipment degrading?** Check for changes under comparable load, while avoiding unsupported fault and remaining-life claims.

### 🌱 Sustainable industrial value
Better condition awareness can help operators **investigate wasteful operation and avoid unnecessary downtime**. Those benefits are possibilities, **not measured savings** in this publisher dataset.

### 🔗 Connection to the three-notebook project
**01 — Forecasting:** Actual-versus-predicted TEC/PV power over the next 15 minutes.  
**02 — Anomaly & Degradation (here):** Data quality, unusual operating behavior, historical review signals.  
**03 — MILP Scheduling:** A *what-if* optimization using forecasts, published prices, real constraints **and the latest known review context**. This notebook does not automatically stop a machine or predict future anomalies.

> **Important:** No verified failure times, maintenance interventions, machine schedules, or run-to-failure trajectories are available. **An anomaly is not a diagnosed fault; a trend is not proof of degradation; remaining useful life (RUL) cannot be estimated here.**

### 🛠️ 1. Setup — how is the analysis kept fair?
We learn reference behavior from the **earliest 60% of quality-valid observations**, calibrate on the next **20%**, and assess the last **20%**. This is a time-ordered split, not a random shuffle.

**Isolation Forest, PCA, Local Outlier Factor and MiniBatch K-Means** compete on earlier historical data. The algorithm and its settings are selected separately for each operating band, then refitted on reference data before calibration. Synthetic changes are only a **proxy test**, not measured fault accuracy. A **2% calibration review budget** sets the statistical threshold; it is not a fault rate. The model compares lower and higher power using the **previous hour**, and a separate rule reviews large machine power transitions. Repeated stable idle readings do not create repeated reviews. Missing measurements remain data-quality issues. I also report the 1% and 5% sensitivity settings.


In [1]:
import os
os.environ.setdefault("POLARS_MAX_THREADS", "4")

from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
import plotly.graph_objects as go
from IPython.display import display, Markdown, HTML
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import make_pipeline

from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from threadpoolctl import threadpool_limits
import warnings

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "etl_config.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the GreenForge AI project.")

OUT = ROOT / "reports" / "ml_short_term" / "notebook_runs"
OUT.mkdir(parents=True, exist_ok=True)
BLUE, ORANGE, GREEN = "#2563eb", "#f59e0b", "#16a34a"
PURPLE, SLATE = "#7c3aed", "#334155"
SEED = 42
INTERVAL_MINUTES = 15  # eda_15min timestamps mark interval START
INGESTION_DELAY_MIN = 0  # minimum known delay; configure from actual production latency
SENSITIVITY_BUDGETS = (0.01, 0.02, 0.05)  # review workload scenarios, not fault probabilities
MIN_COVERAGE = 0.80       # Practical data-quality gate, not a machine limit
REVIEW_SHARE = 0.02       # Fraction of CALIBRATION readings nominated for review
MACHINE_STEP_REVIEW_KW = 0.40  # Review large 15-minute changes; not a fault limit
MIN_TRAIN, MIN_CALIB = 40, 20
ADD_ELECTRICAL_CONTEXT = True  # One selective lazy scan of TEC cleaned Parquet

SYSTEMS = {
    "machine": {"id": "TEC_48S", "label": "Machine power", "active_kw": 0.04},
    "pv": {"id": "IPE_PV", "label": "Solar PV power", "active_kw": 0.20},
}
# Review bands (kW), inferred from the displayed operating levels; validate with site data.
# Machine: <0.04 quiet / >=0.04 energized; PV: <0.20 low / >=0.20 producing.
# Bands describe prior power, not verified on/off states or fault limits.

def report(message):
    display(Markdown("> **💡 What this tells us**\n>\n" +
                     "\n".join("> " + line for line in str(message).strip().splitlines())))

def show_cards(cards):
    blocks = []
    for title, value, note in cards:
        blocks.append(f"""<div style='background:#f8fafc;border:1px solid #e2e8f0;
            border-radius:14px;padding:18px 22px;'>
            <div style='font-size:14px;color:#64748b;margin-bottom:8px;'>{title}</div>
            <div style='font-size:26px;font-weight:700;color:#334155;margin-bottom:6px;'>{value}</div>
            <div style='font-size:13px;color:#475569;'>{note}</div></div>""")
    display(HTML("<div style='display:grid;grid-template-columns:repeat(3,minmax(220px,1fr));gap:14px;margin:12px 0 20px;'>" +
                 "".join(blocks) + "</div>"))

def show_table(frame, columns=None, caption=None):
    table = frame[columns].copy() if columns else frame.copy()
    if caption:
        display(Markdown(f"**{caption}**"))
    if table.empty:
        display(Markdown("*No eligible rows to show.*"))
        return
    fmt = {}
    for col in table.select_dtypes(include="number"):
        fmt[col] = "{:.2f}" if ("pct" in col.lower() or "rate" in col.lower()) else (
            "{:.4f}" if any(key in col.lower() for key in ("score", "surprise", "kw", "delta", "error", "median", "share", "ratio", "threshold", "support"))
            else "{:,.0f}")
    display(table.style.format(fmt, na_rep="—").hide(axis="index").set_table_styles([
        {"selector": "th", "props": [("background-color", "#eff6ff"), ("color", "#1e3a8a"), ("padding", "8px 10px")]},
        {"selector": "td", "props": [("padding", "7px 10px"), ("border-bottom", "1px solid #eef2f7")]},
        {"selector": "table", "props": [("font-family", "Arial, sans-serif"), ("font-size", "13px"), ("border-collapse", "collapse")]},
    ]))

def polish(fig, title, subtitle="", height=430):
    fig.update_layout(title={"text": f"{title}<br><sup>{subtitle}</sup>", "x": 0.05},
        height=height, template="plotly_white", paper_bgcolor="white", plot_bgcolor="white",
        font={"family": "Arial, sans-serif", "color": SLATE, "size": 12},
        margin={"l": 55, "r": 30, "t": 75, "b": 50},
        legend={"orientation": "h", "y": 1.02, "x": 1, "xanchor": "right"},
        hovermode="x unified")
    fig.update_xaxes(gridcolor="#e2e8f0", zeroline=False)
    fig.update_yaxes(gridcolor="#e2e8f0", zeroline=False)
    fig.show()

def local_clock(series):
    t = pd.to_datetime(series)
    return t.dt.tz_convert("Europe/Berlin").dt.tz_localize(None) if t.dt.tz is not None else t

def feature_path(system):
    path = ROOT / "data" / "processed" / "features" / system / "eda_15min.parquet"
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Run the main ETL first.")
    return path


### ❓ 2. Are the measurements trustworthy enough to monitor?
An industrial quality check must come **before** anomaly detection. I report missing power and incomplete 15-minute windows separately; missing PV after its last recorded date is **not** evidence of a solar failure.

For TEC_48S, I optionally add a few measured electrical quantities from `cleaned.parquet`—not all 124 channels. This adds genuine multivariate information without loading the whole raw sensor table into memory.

In [2]:
def load_intervals(key):
    system = SYSTEMS[key]["id"]
    scan = pl.scan_parquet(feature_path(system))
    required = {"WsDateTime", "power_kw", "coverage"}
    missing = required - set(scan.collect_schema().names())
    if missing:
        raise ValueError(f"{system}: missing validated feature columns {sorted(missing)}")
    df = (scan.select("WsDateTime", "power_kw", "coverage")
          .sort("WsDateTime").collect().to_pandas())
    df["WsDateTime"] = local_clock(df["WsDateTime"])
    for col in ("power_kw", "coverage"):
        df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    if df.WsDateTime.duplicated().any():
        raise ValueError(f"{system}: duplicate 15-minute timestamps")
    if not df.WsDateTime.diff().dropna().eq(pd.Timedelta(minutes=15)).all():
        raise ValueError(f"{system}: irregular grid; check the ETL timestamp policy")
    return df


def add_electrical_context(df):
    """Selectively aggregate observed electrical context, including valid sample counts.

    Five-second measurements are used for context only; no ETL artifacts are changed.
    PF_total mean is descriptive, not a physical aggregate power-factor definition.
    """
    path = ROOT / "data" / "processed" / "cleaned" / "TEC_48S" / "cleaned.parquet"
    if not ADD_ELECTRICAL_CONTEXT or not path.is_file():
        return df, []
    scan = pl.scan_parquet(path)
    available = set(scan.collect_schema().names())
    names = {
        "Q_total": "reactive_kvar", "S_total": "apparent_kva",
        "PF_total": "power_factor", "I1": "current_l1_a",
        "I2": "current_l2_a", "I3": "current_l3_a",
        "P_total": "active_kw_sensor",
    }
    names = {source: target for source, target in names.items() if source in available}
    if not names:
        return df, []
    metrics = []
    for source, target in names.items():
        v = pl.col(source).cast(pl.Float64, strict=False)
        valid = pl.when(v.is_finite()).then(v).otherwise(None)
        scale = 1000 if source in ("P_total", "Q_total", "S_total") else 1
        metrics.extend([
            (valid / scale).mean().alias(target),
            valid.count().alias(f"{target}_count"),
        ])
    grouped = (
        scan.select(["WsDateTime", *names])
        .with_columns(pl.col("WsDateTime").dt.truncate("15m").alias("bucket"))
        .group_by("bucket").agg(metrics)
        .rename({"bucket": "WsDateTime"})
        .sort("WsDateTime").collect().to_pandas()
    )
    grouped["WsDateTime"] = local_clock(grouped["WsDateTime"])
    # Keep raw PF for diagnosis, but do not train on the inconsistent PF_total channel.
    extras = [v for v in names.values() if v not in ("active_kw_sensor", "power_factor")]
    return df.merge(grouped, on="WsDateTime", how="left", validate="one_to_one"), extras


def prepare_monitor(key):
    df = load_intervals(key)
    extras = []
    if key == "machine":
        df, extras = add_electrical_context(df)
    df["quality_status"] = np.select(
        [df.power_kw.isna(), df.coverage.isna(), df.coverage < MIN_COVERAGE],
        ["missing_power", "coverage_unknown", "low_coverage"], default="valid"
    )
    # Use recent PAST power to choose the comparison band. Otherwise a surge
    # switches itself into the high-power group and a drop into the low-power group.
    power = df.power_kw.where(df.quality_status == "valid")
    df["previous_kw"] = power.shift(1)
    df["change_kw"] = power - df.previous_kw
    df["past_mean_4"] = power.shift(1).rolling(4, min_periods=2).mean()
    df["past_std_8"] = power.shift(1).rolling(8, min_periods=4).std()
    prior_kw = df.past_mean_4.fillna(df.previous_kw).fillna(power)
    df["state"] = np.where(prior_kw >= SYSTEMS[key]["active_kw"],
                           "higher_load", "lower_load")
    df.loc[df.quality_status != "valid", "state"] = "not_observed"
    df["hour_sin"] = np.sin(2 * np.pi * (df.WsDateTime.dt.hour + df.WsDateTime.dt.minute / 60) / 24)
    df["hour_cos"] = np.cos(2 * np.pi * (df.WsDateTime.dt.hour + df.WsDateTime.dt.minute / 60) / 24)
    # Electrical channels remain available for condition review; the anomaly model
    # monitors power behavior without being dominated by drifting sensor channels.
    candidate_features = ["power_kw", "previous_kw", "change_kw", "past_mean_4",
                          "past_std_8", "hour_sin", "hour_cos"]

    # Split by observed, quality-valid times, preserving the original time order.
    good = np.flatnonzero((df.quality_status == "valid").to_numpy())
    df["period"] = "no_observation"
    if len(good) >= 200:
        a, b = good[int(len(good) * .60)], good[int(len(good) * .80)]
        df.loc[:a - 1, "period"] = "reference"
        df.loc[a:b - 1, "period"] = "calibration"
        df.loc[b:, "period"] = "assessment"
    else:
        df.loc[df.quality_status == "valid", "period"] = "insufficient_history"
    df.loc[df.quality_status != "valid", "period"] = "no_observation"
    return df, candidate_features

monitors, candidate_features = {}, {}
for key in SYSTEMS:
    monitors[key], candidate_features[key] = prepare_monitor(key)

quality_rows = []
for key, df in monitors.items():
    good = df.quality_status.eq("valid")
    quality_rows.append({"Dataset": SYSTEMS[key]["id"], "Intervals": len(df),
        "Usable_intervals": int(good.sum()), "Usable_pct": 100 * good.mean(),
        "Missing_power": int(df.quality_status.eq("missing_power").sum()),
        "Low_coverage": int(df.quality_status.eq("low_coverage").sum()),
        "Last_usable": str(df.loc[good, "WsDateTime"].max()) if good.any() else "None"})
quality_table = pd.DataFrame(quality_rows)
show_table(quality_table, caption="Measurement quality — 15-minute readings")
show_cards([
    ("Machine usable", f"{quality_table.iloc[0].Usable_pct:.1f}%", "valid power and sufficient coverage"),
    ("PV usable", f"{quality_table.iloc[1].Usable_pct:.1f}%", "missing months remain missing"),
    ("Quality gate", f"{MIN_COVERAGE:.0%}", "minimum interval coverage · configurable"),
])
report("Missing readings are **data-quality gaps**, not faults. "
       "Power bands use the previous hour: machine below/above **0.04 kW**; "
       "solar below/above **0.20 kW**. These are review settings, not equipment limits. "
       "Solar history is too short to assess long-term wear.")


**Measurement quality — 15-minute readings**

Dataset,Intervals,Usable_intervals,Usable_pct,Missing_power,Low_coverage,Last_usable
TEC_48S,"35,136","32,919",93.69,"2,217",0,2024-12-31 23:45:00
IPE_PV,"35,136","8,256",23.50,"26,880",0,2024-04-08 23:45:00


> **💡 What this tells us**
>
> Missing readings are **data-quality gaps**, not faults. Power bands use the previous hour: machine below/above **0.04 kW**; solar below/above **0.20 kW**. These are review settings, not equipment limits. Solar history is too short to assess long-term wear.

### ❓ 3. What looks unusual compared with the past?
I compare **higher-load** and **lower-load** behavior separately, using **earlier power readings** to choose the group. A sudden spike or drop is compared with what the system was doing just before it. Large machine power changes also receive a separate **review flag**; this is a screening rule, not a fault diagnosis.

The four unsupervised algorithms compete on earlier reference data using the same proxy test. The winning algorithm and hyperparameters are refitted on reference data. The middle period sets a **review threshold** before evaluating the latest period. Flags are **candidates for a human review**, not diagnosed damage.

In [3]:
# Small, reproducible unsupervised model search.
SEARCH = (
    ("PCA", 2), ("PCA", 4),
    ("KMeans", 8), ("KMeans", 16),
    ("LOF", 20), ("LOF", 40),
    ("Isolation Forest", (96, 128)),
    ("Isolation Forest", (96, 256)),
    ("Isolation Forest", (160, 256)),
)
MAX_REFERENCE = 3000
IDLE_EPS_KW = 1e-6


def fit_candidate(method, parameter, X):
    if method == "LOF":
        model = LocalOutlierFactor(
            n_neighbors=min(parameter, len(X) - 1),
            novelty=True,
            n_jobs=2
        )
    elif method == "Isolation Forest":
        trees, sample_size = parameter
        model = IsolationForest(
            n_estimators=trees,
            max_samples=min(sample_size, len(X)),
            n_jobs=2,
            random_state=SEED
        )
    elif method == "KMeans":
        model = MiniBatchKMeans(
            n_clusters=min(parameter, len(X)),
            batch_size=512,
            n_init=3,
            random_state=SEED
        )
    else:
        model = PCA(
            n_components=min(parameter, X.shape[1], len(X)),
            svd_solver="full"
        )
    if method == "KMeans":
        # Windows MKL + vcomp: avoid excess OpenMP threads in small batches.
        # Filter only the associated scikit-learn warning, not other warnings.
        with threadpool_limits(limits=1, user_api="openmp"):
            with warnings.catch_warnings():
                warnings.filterwarnings(
                    "ignore",
                    message=r"MiniBatchKMeans is known to have a memory leak on Windows with MKL.*",
                    category=UserWarning,
                    module=r"sklearn\.cluster\._kmeans",
                )
                return model.fit(X)
    return model.fit(X)


def anomaly_scores(method, model, X):
    if method in ("LOF", "Isolation Forest"):
        return -model.score_samples(X)

    if method == "KMeans":
        return np.square(model.transform(X)).min(axis=1)

    reconstructed = model.inverse_transform(model.transform(X))
    return np.square(X - reconstructed).mean(axis=1)


def fit_monitor(df, features, key):
    out = df.copy()
    out["surprise_score"] = np.nan
    out["review_flag"] = False
    out["model_flag"] = False
    out["transition_flag"] = False
    out["model_status"] = np.where(
        out.quality_status.eq("valid"),
        "insufficient_reference",
        "data_quality"
    )
    learned = {}

    # Stable machine idle is not a new anomaly.
    stable_idle = (
        out.power_kw.abs().le(IDLE_EPS_KW) &
        out.previous_kw.abs().le(IDLE_EPS_KW)
    ) if key == "machine" else pd.Series(
        False, index=out.index
    )

    # Group consecutive measured PV zeros into episodes.
    # Reset at gaps and local-day boundaries.
    pv_zero = (
        out.quality_status.eq("valid") &
        out.power_kw.abs().le(IDLE_EPS_KW) &
        out.previous_kw.abs().le(IDLE_EPS_KW)
    )

    pv_episode = (
        pv_zero.ne(pv_zero.shift(fill_value=False)) |
        out.WsDateTime.diff().ne(pd.Timedelta(minutes=15)) |
        out.WsDateTime.dt.normalize().ne(
            out.WsDateTime.dt.normalize().shift()
        )
    ).cumsum()

    def collapse_pv_zeros(flags):
        if key != "pv":
            return flags

        zero_flags = flags & pv_zero
        repeated = (
            zero_flags &
            zero_flags.groupby(
                pv_episode, sort=False
            ).cumsum().gt(1)
        )
        return flags & ~repeated

    for state in ("lower_load", "higher_load"):
        ref = out.loc[
            out.period.eq("reference") &
            out.state.eq(state)
        ]
        cal = out.loc[
            out.period.eq("calibration") &
            out.state.eq(state)
        ]

        if len(ref) < MIN_TRAIN or len(cal) < MIN_CALIB:
            continue

        usable = [
            c for c in features
            if ref[c].notna().mean() >= 0.5
        ]
        if len(usable) < 2:
            continue

        # Select the model using reference data only.
        border = max(1, int(len(ref) * 0.80))
        early = ref.iloc[:border]
        validation = ref.iloc[border:]

        if len(early) < 30 or len(validation) < 8:
            continue

        sample = early.iloc[
            np.linspace(
                0, len(early) - 1,
                min(len(early), MAX_REFERENCE),
                dtype=int
            )
        ]

        prep = make_pipeline(
            SimpleImputer(strategy="median"),
            RobustScaler()
        )
        X_train = prep.fit_transform(sample[usable])
        X_val = prep.transform(validation[usable])

        # Simulated changes are proxy tests, not fault labels.
        probes = validation.iloc[
            np.linspace(
                0, len(validation) - 1,
                min(128, len(validation)),
                dtype=int
            )
        ][usable].copy()

        prev = probes["previous_kw"].fillna(
            probes["power_kw"]
        ).fillna(0).to_numpy()

        step = (
            MACHINE_STEP_REVIEW_KW
            if key == "machine" else 1.0
        )
        direction = np.where(
            np.arange(len(probes)) % 2 == 0, 1, -1
        )
        direction = np.where(
            (direction < 0) & (prev < step),
            1, direction
        )

        probes["power_kw"] = np.maximum(
            0, prev + direction * step
        )
        probes["change_kw"] = probes["power_kw"] - prev
        X_probe = prep.transform(probes)

        quiet = (
            (
                validation.power_kw.abs().le(IDLE_EPS_KW) &
                validation.previous_kw.abs().le(IDLE_EPS_KW)
            ).to_numpy()
            if key == "machine"
            else np.zeros(len(validation), dtype=bool)
        )

        # Compare algorithms and hyperparameters.
        results = []

        for method, parameter in SEARCH:
            try:
                candidate = fit_candidate(
                    method, parameter, X_train
                )
                base = anomaly_scores(
                    method, candidate, X_val
                )
                cut = np.quantile(
                    base, 1 - REVIEW_SHARE
                )

                caught = np.mean(
                    anomaly_scores(
                        method, candidate, X_probe
                    ) > cut
                )
                idle_false = (
                    np.mean(base[quiet] > cut)
                    if quiet.any() else 0.0
                )

                # Slightly prefer lighter models.
                cost = {
                    "PCA": 0.0,
                    "KMeans": 0.005,
                    "LOF": 0.01,
                    "Isolation Forest": 0.01
                }[method]

                results.append((
                    caught - 2 * idle_false - cost,
                    method,
                    parameter,
                    float(caught),
                    float(idle_false)
                ))

            except (ValueError, FloatingPointError):
                continue

        if not results:
            continue

        _, method, parameter, proxy_hit, idle_rate = max(
            results, key=lambda x: x[0]
        )

        # Refit the winner on reference data only.
        full_sample = ref.iloc[
            np.linspace(
                0, len(ref) - 1,
                min(len(ref), MAX_REFERENCE),
                dtype=int
            )
        ]

        prep = make_pipeline(
            SimpleImputer(strategy="median"),
            RobustScaler()
        )
        X_ref = prep.fit_transform(full_sample[usable])
        winner = fit_candidate(
            method, parameter, X_ref
        )

        eligible = (
            out.quality_status.eq("valid") &
            out.state.eq(state)
        )

        raw = pd.Series(
            anomaly_scores(
                method,
                winner,
                prep.transform(out.loc[eligible, usable])
            ),
            index=out.index[eligible]
        )

        # Calibrate without using assessment information.
        cal_raw = raw.loc[cal.index]
        actionable = cal_raw.loc[
            ~stable_idle.loc[cal_raw.index]
        ]

        if len(actionable) < MIN_CALIB:
            actionable = cal_raw

        center = float(actionable.median())
        cut_raw = float(np.quantile(
            actionable, 1 - REVIEW_SHARE
        ))
        spread = max(cut_raw - center, 1e-8)

        out.loc[eligible, "surprise_score"] = (
            raw - center
        ) / spread
        out.loc[eligible, "model_status"] = "scored"

        assessment = (
            eligible & out.period.eq("assessment")
        )
        threshold = (cut_raw - center) / spread

        # Statistical anomaly candidates.
        alert = (
            assessment &
            out.surprise_score.gt(threshold) &
            ~stable_idle
        )

        # Keep the first flagged steady-zero reading in each
        # continuous PV episode, per local day and load band.
        alert = collapse_pv_zeros(alert)

        # Preserve the existing machine-transition rule.
        transition = (
            assessment &
            out.change_kw.abs().ge(MACHINE_STEP_REVIEW_KW)
            if key == "machine"
            else pd.Series(False, index=out.index)
        )

        out.loc[alert, "model_flag"] = True
        out.loc[transition, "transition_flag"] = True
        out.loc[
            alert | transition, "review_flag"
        ] = True

        # Apply the same episode policy to sensitivity counts.
        sensitivity = {}

        for budget in SENSITIVITY_BUDGETS:
            cut = (
                np.quantile(actionable, 1 - budget) - center
            ) / spread

            flags = (
                assessment &
                out.surprise_score.gt(cut) &
                ~stable_idle
            )
            flags = collapse_pv_zeros(flags)

            sensitivity[
                f"Review_at_{budget:.0%}_calibration"
            ] = int((flags | transition).sum())

        learned[state] = {
            "method": method,
            "setting": parameter,
            "proxy_hit_pct": round(100 * proxy_hit, 1),
            "idle_false_pct": round(100 * idle_rate, 1),
            "features": usable,
            "reference": len(ref),
            "calibration": len(cal),
            "assessment": int(assessment.sum()),
            "threshold": threshold,
            "calibration_flags": int(
                (actionable > cut_raw).sum()
            ),
            "model_flags": int(alert.sum()),
            "large_changes": int(transition.sum()),
            "assessment_flags": int(
                (alert | transition).sum()
            ),
            "sensitivity": sensitivity,
        }

    # Explain why each reading was flagged.
    out["review_reason"] = np.select(
        [
            out.model_flag & out.transition_flag,
            out.transition_flag,
            out.model_flag
        ],
        [
            "Pattern and large change",
            "Large power change",
            "Unusual pattern"
        ],
        default="none"
    )

    out["review_type"] = np.select(
        [
            out.quality_status.ne("valid"),
            out.review_flag
        ],
        ["data_quality", "unusual_observation"],
        default="none"
    )

    return out, learned


# Fit each monitored system independently.
models = {}

for key in SYSTEMS:
    monitors[key], models[key] = fit_monitor(
        monitors[key],
        candidate_features[key],
        key
    )


# Keep the existing summary structure.
rows, regime_rows = [], []

for key, df in monitors.items():
    assessed = df.loc[
        df.period.eq("assessment") &
        df.model_status.eq("scored")
    ]

    rows.append({
        "Dataset": SYSTEMS[key]["id"],
        "Assessed_intervals": len(assessed),
        "Review_intervals": int(
            assessed.review_flag.sum()
        ),
        "Review_pct": (
            100 * assessed.review_flag.mean()
            if len(assessed) else np.nan
        ),
        "Scored_regimes": ", ".join(models[key]) or "None",
        "Calibration_budget_pct": REVIEW_SHARE * 100,
    })

    for state, d in models[key].items():
        regime_rows.append({
            "Dataset": SYSTEMS[key]["id"],
            "Regime": state,
            "Winner": f"{d['method']} ({d['setting']})",
            "Proxy_hit_pct": d["proxy_hit_pct"],
            "Idle_false_pct": d["idle_false_pct"],
            "Reference": d["reference"],
            "Calibration": d["calibration"],
            "Assessment": d["assessment"],
            "Calibration_flags": d["calibration_flags"],
            "Pattern_flags": d["model_flags"],
            "Large_changes": d["large_changes"],
            "Assessment_flags": d["assessment_flags"],
            **d["sensitivity"],
        })

review_table = pd.DataFrame(rows)
regime_table = pd.DataFrame(regime_rows)

show_table(
    review_table,
    caption="Held-out assessment — readings to review, not confirmed faults"
)

show_table(
    regime_table,
    caption="Selected models and calibration sensitivity (1%, 2%, 5%)"
)

show_cards([
    (
        "Machine review intervals",
        f"{rows[0]['Review_intervals']:,}",
        "held-out readings to check"
    ),
    (
        "PV review intervals",
        f"{rows[1]['Review_intervals']:,}",
        "not diagnosed faults"
    ),
    (
        "Confirmed faults",
        "Unknown",
        "no verified failure labels"
    ),
])

report(
    "Each operating band selects the tuned winner from "
    "**Isolation Forest, LOF, K-Means or PCA** "
    "using simulated historical changes. These proxy tests "
    "are **not measured fault accuracy**. "
    "Stable machine idle is not repeatedly flagged, and "
    f"large machine changes (≥ {MACHINE_STEP_REVIEW_KW:.2f} kW) "
    "remain reviewable. Repeated unusual solar zero readings "
    "are marked once per continuous episode and local day. "
    "All measured readings remain unchanged."
)


**Held-out assessment — readings to review, not confirmed faults**

Dataset,Assessed_intervals,Review_intervals,Review_pct,Scored_regimes,Calibration_budget_pct
TEC_48S,"6,584",44,0.67,"lower_load, higher_load",2.00
IPE_PV,"1,652",53,3.21,"lower_load, higher_load",2.00


**Selected models and calibration sensitivity (1%, 2%, 5%)**

Dataset,Regime,Winner,Proxy_hit_pct,Idle_false_pct,Reference,Calibration,Assessment,Calibration_flags,Pattern_flags,Large_changes,Assessment_flags,Review_at_1%_calibration,Review_at_2%_calibration,Review_at_5%_calibration
TEC_48S,lower_load,PCA (2),100.00,1.20,"17,916",6283.0000,"6,310",1.0000,0,11,11,11.0000,11.0000,11.0000
TEC_48S,higher_load,LOF (40),3.10,0.00,"1,835",301.0000,274,6.0000,2,32,33,32.0000,33.0000,35.0000
IPE_PV,lower_load,PCA (2),100.00,0.00,"3,790",1079.0000,"1,215",22.0000,16,0,16,4.0000,16.0000,20.0000
IPE_PV,higher_load,KMeans (8),24.20,0.00,"1,163",572.0000,437,12.0000,37,0,37,23.0000,37.0000,56.0000


> **💡 What this tells us**
>
> Each operating band selects the tuned winner from **Isolation Forest, LOF, K-Means or PCA** using simulated historical changes. These proxy tests are **not measured fault accuracy**. Stable machine idle is not repeatedly flagged, and large machine changes (≥ 0.40 kW) remain reviewable. Repeated unusual solar zero readings are marked once per continuous episode and local day. All measured readings remain unchanged.

In [4]:
# Optional source-provenance audit: read millions of compressed raw PV rows
# only when the raw input changes or source evidence must be rechecked.

RUN_RAW_PV_AUDIT = False # Set True to rerun the raw PV source audit.

if not RUN_RAW_PV_AUDIT:
    print("Raw PV source audit skipped (set RUN_RAW_PV_AUDIT=True to rerun).")
else:

    import pandas as pd
    import numpy as np

    raw_file = (
        ROOT / "data/raw/spark_raw/IPE_PV"
        / "AC_ActivePower/2024_AC_ActivePower.csv.xz"
    )

    periods = {
        "Before April 4": ("2024-01-01", "2024-04-04"),
        "April 4–8": ("2024-04-04", "2024-04-09"),
        "April 9 onward": ("2024-04-09", "2025-01-01"),
    }

    stats = {
        name: {
            "Rows": 0,
            "Valid": 0,
            "Zeros": 0,
            "Positive": 0,
            "Missing": 0,
            "Last_valid": pd.NaT
        }
        for name in periods
    }

    first_raw = pd.NaT
    last_raw = pd.NaT

    for chunk in pd.read_csv(
        raw_file,
        compression="xz",
        usecols=["WsDateTime", "006_AC_ActivePower"],
        chunksize=250_000
    ):
        times = pd.to_datetime(
            chunk["WsDateTime"], errors="coerce"
        )

        power = pd.to_numeric(
            chunk["006_AC_ActivePower"], errors="coerce"
        )

        finite = pd.Series(
            np.isfinite(power.to_numpy()),
            index=chunk.index
        )

        if times.notna().any():
            tmin, tmax = times.min(), times.max()

            if pd.isna(first_raw) or tmin < first_raw:
                first_raw = tmin

            if pd.isna(last_raw) or tmax > last_raw:
                last_raw = tmax

        for name, (start, end) in periods.items():
            mask = times.ge(start) & times.lt(end)
            valid = mask & finite

            s = stats[name]
            s["Rows"] += int(mask.sum())
            s["Valid"] += int(valid.sum())
            s["Zeros"] += int((valid & power.eq(0)).sum())
            s["Positive"] += int((valid & power.gt(0)).sum())
            s["Missing"] += int((mask & ~finite).sum())

            if valid.any():
                latest = times.loc[valid].max()
                if pd.isna(s["Last_valid"]) or latest > s["Last_valid"]:
                    s["Last_valid"] = latest

    print("RAW SOLAR POWER — SOURCE AVAILABILITY")
    print("First raw timestamp:", first_raw)
    print("Last raw timestamp: ", last_raw)

    show_table(
        pd.DataFrame([
            {"Period": name, **values}
            for name, values in stats.items()
        ]),
        caption="Raw solar PV — original power measurements"
    )


Raw PV source audit skipped (set RUN_RAW_PV_AUDIT=True to rerun).


### ❓ 4. When should an engineer look more closely?
A single unusual quarter-hour is not enough to declare a machine faulty. We show **episodes of adjacent review intervals** and a **real measured-power graph** so an engineer can see what happened around them.

In [5]:
def review_episodes(frame, system):
    flags = frame.loc[frame.review_flag, ["WsDateTime", "power_kw", "surprise_score"]].copy()
    if flags.empty:
        return pd.DataFrame(columns=["System", "Start", "End", "Duration_min", "Intervals", "Peak_abs_kW", "Peak_surprise"])
    episode = flags.WsDateTime.diff().ne(pd.Timedelta(minutes=15)).cumsum()
    out = (flags.assign(_episode=episode)
           .groupby("_episode", as_index=False)
           .agg(Start=("WsDateTime", "min"), Last=("WsDateTime", "max"),
                Intervals=("WsDateTime", "size"), Peak_abs_kW=("power_kw", lambda s: s.abs().max()),
                Peak_surprise=("surprise_score", "max")))
    out["End"] = out.pop("Last") + pd.Timedelta(minutes=15)
    out["Duration_min"] = out["Intervals"] * 15
    out["System"] = system
    return out[["System", "Start", "End", "Duration_min", "Intervals", "Peak_abs_kW", "Peak_surprise"]]

all_events = pd.concat([review_episodes(df, SYSTEMS[key]["id"])
                        for key, df in monitors.items()], ignore_index=True)
events = all_events.sort_values(["Peak_surprise", "Duration_min"], ascending=False)
show_table(events.head(12), caption="Top review episodes — grouped to reduce alert noise")
if len(events):
    concentration = 100 * events.head(5)["Intervals"].sum() / events["Intervals"].sum()
    display(Markdown(
        f"**Alert concentration:** the five highest-surprise episodes "
        f"contain **{concentration:.1f}%** of reviewed intervals. "
        "These are observations for human investigation, not measured downtime."
    ))


def plot_review_window(key):
    df = monitors[key]
    assessed = df.loc[(df.period == "assessment") & df.power_kw.notna()].copy()
    if assessed.empty:
        display(Markdown(f"*{SYSTEMS[key]['label']}: no observed assessment window to plot.*"))
        return
    alerts = assessed.loc[assessed.review_flag]
    # Show the highest assessed machine load, not the largest shutdown.
    if key == "machine":
        focus = assessed.loc[assessed.power_kw.idxmax(), "WsDateTime"]
    else:
        focus = (alerts.loc[alerts.change_kw.abs().fillna(0).idxmax(), "WsDateTime"] if len(alerts)
                 else assessed.loc[assessed.power_kw.idxmax(), "WsDateTime"])
    view = df.loc[df.WsDateTime.between(focus - pd.Timedelta(hours=24),
                                        focus + pd.Timedelta(hours=24))].copy()
    bad = view.loc[view.review_flag]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=view.WsDateTime, y=view.power_kw, name="Measured power",
        mode="lines", line=dict(color=BLUE, width=2.5), connectgaps=False,
        hovertemplate="%{x|%d %b %Y %H:%M}<br>%{y:.3f} kW<extra></extra>"))
    if not bad.empty:
        fig.add_trace(go.Scatter(x=bad.WsDateTime, y=bad.power_kw,
            mode="markers", name="Review this reading", marker=dict(color=ORANGE,
            size=10, symbol="diamond", line=dict(color="white", width=1)),
            hovertemplate="%{x|%d %b %Y %H:%M}<br>%{y:.3f} kW<extra></extra>"))
    fig.update_yaxes(title="Observed 15-minute mean power (kW)")
    fig.update_xaxes(title="Berlin local wall-clock time")
    # Optional zoom changes only the axis, not the measured series or review flags.
    low_top = 0.12 if key == "machine" else 0.40
    fig.update_layout(updatemenus=[dict(type="buttons", direction="right", x=0, y=1.03,
        buttons=[dict(label="Full range", method="relayout", args=[{"yaxis.autorange": True}]),
                 dict(label="Low-power detail", method="relayout",
                      args=[{"yaxis.range": [-0.005, low_top]}])])])
    polish(fig, f"Measured behavior and review flags — {SYSTEMS[key]['label']}",
           "48-hour context; only observed assessment-period anomalies are marked", height=425)

plot_review_window("machine")
plot_review_window("pv")
report(f"**{len(events):,} review episodes** across both systems. "
       "The machine view shows its **highest assessed 15-minute mean**, not the EDA five-second peak. "
       "Orange means **check this reading**, not a confirmed failure. "
       "Use **Low-power detail** to see small changes without hiding larger peaks. "
       "Missing measurements remain gaps.")


**Top review episodes — grouped to reduce alert noise**

System,Start,End,Duration_min,Intervals,Peak_abs_kW,Peak_surprise
IPE_PV,2024-03-23 16:30:00,2024-03-23 17:00:00,30.0000,2,0.0896,10.3757
IPE_PV,2024-04-02 17:15:00,2024-04-02 18:00:00,45.0000,3,0.0239,5.9867
IPE_PV,2024-03-25 11:00:00,2024-03-25 12:00:00,60.0000,4,5.7789,5.9275
IPE_PV,2024-03-31 11:00:00,2024-03-31 11:30:00,30.0000,2,6.4460,4.9348
IPE_PV,2024-03-28 09:30:00,2024-03-28 09:45:00,15.0000,1,5.4724,4.4833
IPE_PV,2024-03-24 12:15:00,2024-03-24 12:30:00,15.0000,1,5.3941,4.0633
IPE_PV,2024-03-25 17:15:00,2024-03-25 18:00:00,45.0000,3,0.0259,3.7609
IPE_PV,2024-03-28 13:00:00,2024-03-28 13:15:00,15.0000,1,1.5639,2.9009
IPE_PV,2024-03-25 10:30:00,2024-03-25 10:45:00,15.0000,1,5.1623,2.6635
IPE_PV,2024-03-29 09:00:00,2024-03-29 09:15:00,15.0000,1,0.9587,2.6011


**Alert concentration:** the five highest-surprise episodes contain **12.4%** of reviewed intervals. These are observations for human investigation, not measured downtime.

> **💡 What this tells us**
>
> **56 review episodes** across both systems. The machine view shows its **highest assessed 15-minute mean**, not the EDA five-second peak. Orange means **check this reading**, not a confirmed failure. Use **Low-power detail** to see small changes without hiding larger peaks. Missing measurements remain gaps.

In [6]:
# Where do unusual readings occur?
# Only scored assessment readings; no model changes.

def plot_anomaly_scatter(key):
    df = monitors[key]

    data = df.loc[
        df.period.eq("assessment")
        & df.model_status.eq("scored")
        & df.quality_status.eq("valid")
        & df.power_kw.notna()
        & df.change_kw.notna()
    ].copy()

    if data.empty:
        display(Markdown(
            f"*{SYSTEMS[key]['label']}: no eligible readings.*"
        ))
        return

    fig = go.Figure()

    for flagged, label, color, size, opacity in (
        (False, "Not flagged", BLUE, 5, 0.35),
        (True, "Review flagged", ORANGE, 10, 1.0),
    ):
        subset = data.loc[data.review_flag.eq(flagged)]

        if subset.empty:
            continue

        hover = np.column_stack([
            subset.WsDateTime.dt.strftime("%d %b %Y %H:%M"),
            subset.state,
            subset.review_reason
        ])

        fig.add_trace(go.Scattergl(
            x=subset.power_kw,
            y=subset.change_kw,
            mode="markers",
            name=label,
            customdata=hover,
            marker=dict(
                color=color,
                size=size,
                opacity=opacity,
                symbol="diamond" if flagged else "circle"
            ),
            hovertemplate=(
                "<b>%{customdata[0]}</b>"
                "<br>Power: %{x:.3f} kW"
                "<br>Change: %{y:+.3f} kW"
                "<br>Band: %{customdata[1]}"
                "<br>Reason: %{customdata[2]}"
                "<extra></extra>"
            )
        ))

    # Match the notebook's existing Plotly style.
    fig.update_layout(
        title=dict(
            text=(
                f"Power behavior and review flags — "
                f"{SYSTEMS[key]['label']}"
                "<br><sup>Scored assessment readings · "
                "not confirmed equipment faults</sup>"
            ),
            x=0.05
        ),
        template="plotly_white",
        height=440,
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=dict(
            family="Arial, sans-serif",
            color=SLATE,
            size=12
        ),
        margin=dict(l=65, r=30, t=90, b=60),
        legend=dict(
            orientation="h",
            y=1.02,
            x=1,
            xanchor="right"
        ),
        hovermode="closest"
    )

    fig.update_xaxes(
        title="Observed 15-minute mean power (kW)",
        gridcolor="#e2e8f0",
        zeroline=False
    )

    fig.update_yaxes(
        title="15-minute power change (kW)",
        gridcolor="#e2e8f0",
        zeroline=True,
        zerolinecolor="#cbd5e1"
    )

    fig.show()

    reviews = int(data.review_flag.sum())
    display(Markdown(
        f"**{SYSTEMS[key]['label']}:** "
        f"{len(data):,} plotted readings · "
        f"**{reviews:,} flagged for review** "
        f"({reviews / len(data):.2%})."
    ))


plot_anomaly_scatter("machine")
plot_anomaly_scatter("pv")

report(
    "Blue points were not flagged; orange points need review. "
    "The plots show measured power versus its 15-minute change. "
    "Only valid, scored assessment readings with an available "
    "power-change value are plotted. Overlapping points may "
    "hide repeated readings. Flags are not confirmed faults."
)


**Machine power:** 6,505 plotted readings · **44 flagged for review** (0.68%).

**Solar PV power:** 1,652 plotted readings · **53 flagged for review** (3.21%).

> **💡 What this tells us**
>
> Blue points were not flagged; orange points need review. The plots show measured power versus its 15-minute change. Only valid, scored assessment readings with an available power-change value are plotted. Overlapping points may hide repeated readings. Flags are not confirmed faults.

### ❓ 5. Does the forecasting notebook support these observations?
Notebook 01 exports `machine_predictions_15min.csv` and `pv_predictions_15min.csv` in `reports/ml_short_term/notebook_runs`. I use them to **compare actual and predicted values at the same target timestamp**. That is a retrospective diagnostic: a forecast error becomes known **only after** the actual next interval is measured.

A large forecasting error can reflect an unexpected start, a weather-driven solar change, or simply an imperfect model. Notebook 01 currently highlights test-R² winners **after comparing models on the test period**; that is an exploratory benchmark, not a fully independent deployment guarantee. **Forecast errors are not fault labels or future scheduling constraints.**

In [7]:
def attach_forecasts(df, key):
    out = df.copy()
    out["forecast_kw"] = np.nan
    out["forecast_error_kw"] = np.nan
    path = OUT / f"{key}_predictions_15min.csv"
    if not path.is_file():
        return out, "Forecast export not found; run Notebook 01 first."
    forecast = pd.read_csv(path, usecols=["target_time", "target", "prediction"])
    forecast["target_time"] = local_clock(forecast["target_time"])
    if forecast.target_time.duplicated().any():
        raise ValueError(f"{path.name}: duplicate forecast target timestamps")
    for column in ("target", "prediction"):
        forecast[column] = pd.to_numeric(forecast[column], errors="coerce")
    merged = out.merge(forecast, left_on="WsDateTime", right_on="target_time",
                       how="left", validate="one_to_one")
    comparison = merged.loc[merged.target.notna() & merged.power_kw.notna()]
    if len(comparison) and (comparison.target - comparison.power_kw).abs().max() > 1e-3:
        raise ValueError(f"{key}: Notebook 01 targets do not match the current ETL power values")
    # A post-observation diagnostic; never present this as a known future risk.
    merged["forecast_kw"] = merged["prediction"]
    merged["forecast_error_kw"] = (merged.power_kw - merged.prediction).abs()
    merged.drop(columns=["target_time", "target", "prediction"], inplace=True)
    return merged, f"{int(merged.forecast_error_kw.notna().sum()):,} matching forecast intervals"

forecast_notes = {}
for key in SYSTEMS:
    monitors[key], forecast_notes[key] = attach_forecasts(monitors[key], key)

forecast_rows = []
for key, df in monitors.items():
    x = df.loc[df.forecast_error_kw.notna()]
    reviewed = x.loc[x.review_flag]
    forecast_rows.append({"Dataset": SYSTEMS[key]["id"],
        "Matched_intervals": len(x),
        "Forecast_MAE_kW": x.forecast_error_kw.mean() if len(x) else np.nan,
        "Review_MAE_kW": reviewed.forecast_error_kw.mean() if len(reviewed) else np.nan,
        "Review_matches": len(reviewed), "Source": forecast_notes[key]})
show_table(pd.DataFrame(forecast_rows), caption="Forecast error as additional historical context")
report("Forecast differences are evidence about **prediction difficulty**, not equipment damage. "
       "The comparison is aligned with the same target interval recorded by Notebook 01. "
       "There is no price or weather prediction here, and no future actual reading is used in a present-time alert.")

**Forecast error as additional historical context**

Dataset,Matched_intervals,Forecast_MAE_kW,Review_MAE_kW,Review_matches,Source
TEC_48S,"6,584",0.0093,0.6182,44,"6,584 matching forecast intervals"
IPE_PV,"1,651",0.1755,1.2131,53,"1,651 matching forecast intervals"


> **💡 What this tells us**
>
> Forecast differences are evidence about **prediction difficulty**, not equipment damage. The comparison is aligned with the same target interval recorded by Notebook 01. There is no price or weather prediction here, and no future actual reading is used in a present-time alert.

### ❓ 6. Is there evidence of wear or gradual degradation?
A large change in power factor deserves investigation, but **it is not proof of damage**. The saved results show that `PF_total` and the independent P/S ratio disagree in early data. I therefore compare **P/S first**, while keeping raw power factor visible for sensor checks.

I compare **early and late, relatively stable readings within the same measured-power band**, report sample support, and cross-check reactive power, apparent power and phase currents. The current comparable band may represent **low-load or standby operation**, not verified machining. Sensor disagreements are **checks to investigate**, not proof of wear. Actual physical degradation detection **remains unvalidated**; no degradation rate or remaining useful life is estimated.


In [8]:
# Electrical condition review — not a degradation diagnosis.
machine = monitors["machine"].copy()
min_sensor_samples = int(
    np.ceil(MIN_COVERAGE * INTERVAL_MINUTES * 60 / 5)
)


def checked_sensor(name):
    """Use finite sensor readings with sufficient coverage."""
    if name not in machine:
        return pd.Series(np.nan, index=machine.index, dtype=float)

    values = pd.to_numeric(
        machine[name], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

    count = f"{name}_count"
    return (
        values.where(machine[count] >= min_sensor_samples)
        if count in machine else values
    )


# 1. Electrical indicators from observed measurements.
s = checked_sensor("apparent_kva").where(lambda x: x > 0.02)
q = checked_sensor("reactive_kvar")

machine["pf_from_power"] = (
    machine.power_kw.abs() / s
).where(lambda x: x.between(0, 1.05))

machine["reactive_share"] = (
    q.abs() / s
).where(lambda x: x.between(0, 1.05))

machine["power_factor"] = checked_sensor("power_factor")

currents = [
    "current_l1_a",
    "current_l2_a",
    "current_l3_a"
]

for column in currents:
    machine[column] = checked_sensor(column)

phase = machine[currents].abs()

machine["current_imbalance_pct"] = (
    100 * (phase.max(axis=1) - phase.min(axis=1))
    / phase.mean(axis=1)
).where(
    phase.notna().all(axis=1)
    & phase.min(axis=1).gt(0.05)
)

condition_result = "Inconclusive — insufficient comparable measurements"
condition_table = pd.DataFrame()
pf_disagreement = np.nan


# 2. Compare valid, relatively stable electrical loads.
stable = (
    machine.quality_status.eq("valid")
    & machine.state.eq("higher_load")
    & machine.power_kw.gt(SYSTEMS["machine"]["active_kw"])
    & machine.previous_kw.notna()
    & machine.change_kw.abs().le(
        np.maximum(0.02, 0.20 * machine.power_kw.abs())
    )
    & machine.pf_from_power.notna()
)

base = machine.loc[
    stable & machine.period.eq("reference")
]

recent = machine.loc[
    stable & machine.period.eq("assessment")
]

if len(base) >= 30 and len(recent) >= 10:

    lo, hi = base.power_kw.quantile([0.25, 0.75])

    early = base.loc[
        base.power_kw.between(lo, hi)
    ]

    late = recent.loc[
        recent.power_kw.between(lo, hi)
    ]

    if len(early) >= 20 and len(late) >= 10:

        power_delta = abs(
            early.power_kw.median() - late.power_kw.median()
        )

        comparable = power_delta <= max(
            0.01, 0.10 * early.power_kw.median()
        )

        band_label = (
            f"{lo:.3f}–{hi:.3f} kW "
            f"({lo * 1000:.0f}–{hi * 1000:.0f} W)"
        )

        display(Markdown(
            f"**Compared electrical load:** {band_label} · "
            f"**{len(early):,} early / {len(late):,} later** "
            f"intervals · median power difference "
            f"**{power_delta:.3f} kW**. "
            "Similar electrical power does **not** establish "
            "identical machining conditions."
        ))

        # 3. Compare available electrical measurements.
        measures = [
            "power_kw",
            "pf_from_power",
            "reactive_share",
            "power_factor",
            "reactive_kvar",
            "apparent_kva",
            *currents,
            "current_imbalance_pct"
        ]

        rows = []

        for column in measures:
            a = early[column].dropna()
            b = late[column].dropna()

            if a.empty and b.empty:
                continue

            old = a.median() if len(a) else np.nan
            new = b.median() if len(b) else np.nan

            rows.append({
                "Measure": column,
                "Early_n": len(a),
                "Later_n": len(b),
                "Early_median": old,
                "Later_median": new,
                "Delta": new - old
            })

        condition_table = pd.DataFrame(rows)

        show_table(
            condition_table,
            caption="Electrical indicators — stable-load comparison"
        )

        # 4. Weekly comparison reduces repeated-reading influence.
        compare = pd.concat([
            early.assign(Stage="Early reference"),
            late.assign(Stage="Later assessment")
        ], ignore_index=True)

        compare["Week"] = (
            compare.WsDateTime.dt.to_period("W").dt.start_time
        )

        weekly = (
            compare.groupby(
                ["Week", "Stage"], as_index=False
            )
            .agg(
                Median=("pf_from_power", "median"),
                Count=("pf_from_power", "count")
            )
        )

        weekly = weekly.loc[weekly.Count >= 5]

        w_early = weekly.loc[
            weekly.Stage.eq("Early reference"), "Median"
        ]

        w_late = weekly.loc[
            weekly.Stage.eq("Later assessment"), "Median"
        ]

        # Descriptive screening only, not a degradation test.
        if not comparable:
            finding = (
                "Inconclusive — different power distributions"
            )

        elif len(w_early) < 3 or len(w_late) < 3:
            finding = (
                "Inconclusive — insufficient supported weeks"
            )

        else:
            baseline = w_early.median()
            shift = w_late.median() - baseline

            mad = (w_early - baseline).abs().median()
            margin = max(0.02, 3 * 1.4826 * mad)

            persistent = (
                (w_late > baseline + margin).mean() >= 0.75
                if shift > 0
                else (w_late < baseline - margin).mean() >= 0.75
            )

            finding = (
                "Persistent P/S difference observed"
                if abs(shift) > margin and persistent
                else "No clear persistent P/S difference"
            )

            finding += (
                f" (weekly shift {shift:+.3f}; "
                f"{len(w_early)} early / "
                f"{len(w_late)} later weeks)"
            )

        condition_result = (
            f"{finding} at {band_label}; "
            "overall machine wear not assessed"
        )

        # 5. Preserve the original Plotly graph.
        fig = go.Figure()

        for stage, color in (
            ("Early reference", BLUE),
            ("Later assessment", ORANGE)
        ):
            part = weekly.loc[
                weekly.Stage.eq(stage)
            ]

            fig.add_trace(go.Scatter(
                x=part.Week,
                y=part.Median,
                name=stage,
                mode="lines+markers",
                line=dict(color=color, width=2.5),
                marker=dict(size=6),
                connectgaps=False
            ))

        fig.update_yaxes(
            title="Weekly median pf_from_power"
        )
        fig.update_xaxes(
            title="Berlin local date"
        )

        polish(
            fig,
            "Electrical behavior at comparable measured power",
            "Stable-load comparison · not a fault or lifetime diagnosis",
            height=420
        )

        # 6. Cross-check recorded power factor against P/S.
        early_gap = (
            early.power_factor - early.pf_from_power
        ).abs().dropna()

        late_gap = (
            late.power_factor - late.pf_from_power
        ).abs().dropna()

        if len(early_gap):
            pf_disagreement = float(early_gap.median())

        if (
            len(early_gap)
            and len(late_gap)
            and early_gap.median() > 0.20
        ):
            display(Markdown(
                "**Power-factor sensor check:** "
                "Median |recorded PF − P/S| is "
                f"**{early_gap.median():.3f} early** versus "
                f"**{late_gap.median():.3f} later**. "
                "Verify measurement definitions and sensor "
                "configuration before interpreting this as wear."
            ))

        # 7. Check phase-3 sensor consistency.
        i3_early = early.current_l3_a.median()
        i3_late = late.current_l3_a.median()

        if (
            pd.notna(i3_early)
            and pd.notna(i3_late)
            and i3_early > 1
            and i3_late < 0.05
        ):
            display(Markdown(
                "**Phase-3 sensor check:** Current changes from "
                f"**{i3_early:.2f} A to {i3_late:.2f} A**. "
                "Later three-phase imbalance cannot be "
                "assessed reliably. Check channel mapping, "
                "units and calibration. This is **not proof of wear**."
            ))

    else:
        condition_result = (
            "Inconclusive — too few observations at the same "
            f"stable load ({len(early)} early / {len(late)} later)"
        )

else:
    condition_result = (
        "Inconclusive — too few valid stable P/S measurements "
        f"({len(base)} early / {len(recent)} later)"
    )


# 8. Clear and scientifically cautious conclusion.
display(Markdown(
    f"**Condition-monitoring status:** {condition_result}."
))

report(
    "This compares electrical behavior at one measured "
    "power band, not machine health across all operating loads. "
    "Without verified machining states, sensor checks and "
    "maintenance records, physical degradation and remaining "
    "useful life cannot be established. The 0.02 P/S screening "
    "margin is descriptive, not a fault threshold."
)


display(Markdown(
    "**Degradation validation: NOT VALIDATED** — "
    "The electrical comparison found no clear persistent P/S change "
    "at the selected low-power operating band. However, this does "
    "not confirm machine health or the absence of wear. "
    "Verified operating states, maintenance records and degradation "
    "labels are unavailable for independent validation. "
    "**Physical degradation, fault-detection accuracy and remaining "
    "useful life cannot be established from this analysis.**"
))


**Compared electrical load:** 0.067–0.069 kW (67–69 W) · **980 early / 121 later** intervals · median power difference **0.001 kW**. Similar electrical power does **not** establish identical machining conditions.

**Electrical indicators — stable-load comparison**

Measure,Early_n,Later_n,Early_median,Later_median,Delta
power_kw,980,121,0.0672,0.0678,0.0007
pf_from_power,980,121,0.6659,0.6579,-0.0080
reactive_share,980,121,0.7463,0.7533,0.0070
power_factor,980,121,1.0000,0.6579,-0.3421
reactive_kvar,980,121,0.0755,0.0777,0.0023
apparent_kva,980,121,0.1011,0.1032,0.0020
current_l1_a,980,121,0.2389,0.2405,0.0016
current_l2_a,980,121,0.2326,0.2372,0.0046
current_l3_a,980,121,11.0941,0.0000,-11.0941
current_imbalance_pct,856,0,281.7995,—,—


**Power-factor sensor check:** Median |recorded PF − P/S| is **0.327 early** versus **0.000 later**. Verify measurement definitions and sensor configuration before interpreting this as wear.

**Phase-3 sensor check:** Current changes from **11.09 A to 0.00 A**. Later three-phase imbalance cannot be assessed reliably. Check channel mapping, units and calibration. This is **not proof of wear**.

**Condition-monitoring status:** No clear persistent P/S difference (weekly shift -0.004; 6 early / 4 later weeks) at 0.067–0.069 kW (67–69 W); overall machine wear not assessed.

> **💡 What this tells us**
>
> This compares electrical behavior at one measured power band, not machine health across all operating loads. Without verified machining states, sensor checks and maintenance records, physical degradation and remaining useful life cannot be established. The 0.02 P/S screening margin is descriptive, not a fault threshold.

**Degradation validation: NOT VALIDATED** — The electrical comparison found no clear persistent P/S change at the selected low-power operating band. However, this does not confirm machine health or the absence of wear. Verified operating states, maintenance records and degradation labels are unavailable for independent validation. **Physical degradation, fault-detection accuracy and remaining useful life cannot be established from this analysis.**

### ❓ 7. What should go to the scheduling notebook?
I export one **historical** anomaly/quality table for Notebook 03. Each `WsDateTime` marks the **start** of a 15-minute interval, while the anomaly score and forecast error can only be known **after that interval finishes**.

The handoff therefore includes `interval_end` and **`available_at`**. The scheduler must filter using `available_at <= decision_time`, not `WsDateTime <= decision_time`. The optional processing delay is set to zero as a minimum; real deployment must include verified ingestion and scoring latency.

The flags are **advisory**, not automatic machine-stop commands. A review recommendation is not a verified machine-safety restriction.


In [9]:
exports = []
for key, df in monitors.items():
    selected = ["WsDateTime", "power_kw", "coverage", "quality_status", "state",
                "period", "model_status", "surprise_score", "review_flag",
                "review_type", "review_reason", "forecast_error_kw"]
    prepared = df[selected].copy()
    prepared.insert(1, "system", SYSTEMS[key]["id"])

    # eda_15min timestamps identify the START of their measurement interval.
    prepared["interval_end"] = (prepared["WsDateTime"] +
                                pd.Timedelta(minutes=INTERVAL_MINUTES))
    prepared["available_at"] = (prepared["interval_end"] +
                                pd.Timedelta(minutes=INGESTION_DELAY_MIN))
    prepared["advisory"] = np.select(
        [prepared.quality_status.ne("valid"),
         prepared.period.ne("assessment") | prepared.model_status.ne("scored"),
         prepared.review_flag],
        ["measurement_unavailable", "not_assessed", "review_observation"],
        default="no_model_alert",
    )
    exports.append(prepared)

handoff = (pd.concat(exports, ignore_index=True)
           .sort_values(["available_at", "system"]).reset_index(drop=True))
assert (handoff.available_at >= handoff.interval_end).all()
assert (handoff.interval_end > handoff.WsDateTime).all()

# Example for Notebook 03: take the latest record ALREADY AVAILABLE at a decision.
def latest_known_context(data, system, decision_time):
    past = data.loc[(data.system == system) &
                    (data.available_at <= pd.Timestamp(decision_time))]
    if past.empty:
        return pd.DataFrame(columns=data.columns)
    return past.tail(1).copy()

# Sanity check: a decision at an interval START cannot see that interval's result.
example = handoff.loc[handoff.system == "TEC_48S"].tail(1)
if not example.empty:
    decision_time = example.iloc[0].WsDateTime
    known = latest_known_context(handoff, "TEC_48S", decision_time)
    assert known.empty or known.iloc[-1].available_at <= decision_time
    assert known.empty or known.iloc[-1].WsDateTime < decision_time

handoff.to_parquet(OUT / "anomaly_review_15min.parquet", index=False)
events.to_csv(OUT / "anomaly_review_episodes.csv", index=False)
quality_table.to_csv(OUT / "anomaly_data_quality.csv", index=False)
review_table.to_csv(OUT / "anomaly_review_summary.csv", index=False)

show_table(pd.DataFrame([
    {"Output": "anomaly_review_15min.parquet", "Use": "Available-at aligned advisory for Notebook 03"},
    {"Output": "anomaly_review_episodes.csv", "Use": "Grouped observations for human review"},
    {"Output": "anomaly_data_quality.csv", "Use": "Valid and missing observation coverage"},
    {"Output": "anomaly_review_summary.csv", "Use": "Held-out assessment alert rates"},
]), caption="Outputs saved to reports/ml_short_term/notebook_runs")
report(
    "**Time-safe handoff:** Each row is usable only when `available_at` is "
    "no later than the scheduling decision time. A 10:00–10:15 reading cannot "
    "influence a decision made at 10:00. The exported `forecast_error_kw` is "
    "also retrospective and must not be used as a future known outcome. "
    "Only verified operating/safety constraints can authorize a scheduling block."
)


**Outputs saved to reports/ml_short_term/notebook_runs**

Output,Use
anomaly_review_15min.parquet,Available-at aligned advisory for Notebook 03
anomaly_review_episodes.csv,Grouped observations for human review
anomaly_data_quality.csv,Valid and missing observation coverage
anomaly_review_summary.csv,Held-out assessment alert rates


> **💡 What this tells us**
>
> **Time-safe handoff:** Each row is usable only when `available_at` is no later than the scheduling decision time. A 10:00–10:15 reading cannot influence a decision made at 10:00. The exported `forecast_error_kw` is also retrospective and must not be used as a future known outcome. Only verified operating/safety constraints can authorize a scheduling block.

In [10]:

# 🛠️ Historical TEC_48S assessment for Notebook 03
# Training and calibration use only earlier machine readings.
# The original Notebook 02 evaluation remains unchanged.

EARLY_TEST_START = pd.Timestamp("2024-03-24 08:00:00")
EARLY_TEST_END = pd.Timestamp("2024-04-09 00:00:00")

EARLY_MONITOR_FILE = (
    OUT / "machine_monitoring_early_assessment_15min.parquet"
)

machine_early = monitors["machine"].copy()

valid_before = machine_early.index[
    machine_early["WsDateTime"].lt(EARLY_TEST_START)
    & machine_early["quality_status"].eq("valid")
]

if len(valid_before) < 200:
    raise ValueError(
        "Not enough valid machine readings before the test period."
    )

# Chronological pre-test split:
# 70% reference, 30% calibration.
cut = int(0.70 * len(valid_before))

machine_early["period"] = "no_observation"

machine_early.loc[
    valid_before[:cut], "period"
] = "reference"

machine_early.loc[
    valid_before[cut:], "period"
] = "calibration"

# Later measurements are used only for assessment.
in_test = (
    machine_early["WsDateTime"].ge(EARLY_TEST_START)
    & machine_early["WsDateTime"].lt(EARLY_TEST_END)
    & machine_early["quality_status"].eq("valid")
)

machine_early.loc[in_test, "period"] = "assessment"

# Reuse Notebook 02's existing model-search function.
machine_early, early_models = fit_monitor(
    machine_early,
    candidate_features["machine"],
    "machine"
)

# Prepare the time-aware handoff.
export_early = machine_early.loc[
    machine_early["WsDateTime"].ge(EARLY_TEST_START)
    & machine_early["WsDateTime"].lt(EARLY_TEST_END),
    [
        "WsDateTime",
        "power_kw",
        "coverage",
        "quality_status",
        "period",
        "model_status",
        "review_flag",
        "review_reason",
    ],
].copy()

export_early.insert(1, "system", "TEC_48S")

export_early["interval_end"] = (
    export_early["WsDateTime"]
    + pd.Timedelta(minutes=INTERVAL_MINUTES)
)

export_early["available_at"] = (
    export_early["interval_end"]
    + pd.Timedelta(minutes=INGESTION_DELAY_MIN)
)

export_early["advisory"] = np.select(
    [
        export_early["quality_status"].ne("valid"),
        export_early["period"].ne("assessment")
        | export_early["model_status"].ne("scored"),
        export_early["review_flag"],
    ],
    [
        "measurement_unavailable",
        "not_assessed",
        "review_observation",
    ],
    default="no_model_alert",
)

assert (
    export_early["available_at"] >= export_early["interval_end"]
).all()

export_early.to_parquet(
    EARLY_MONITOR_FILE, index=False
)

assessed_early = export_early.loc[
    export_early["period"].eq("assessment")
    & export_early["model_status"].eq("scored")
]

show_table(
    pd.DataFrame([{
        "System": "TEC_48S",
        "Pre-test valid readings": len(valid_before),
        "Scored test intervals": len(assessed_early),
        "Review observations": int(
            assessed_early["review_flag"].sum()
        ),
        "Models trained": ", ".join(early_models) or "None",
    }]),
    caption="Separate historical machine assessment"
)

report(
    f"The machine detector learned only from readings before "
    f"**{EARLY_TEST_START:%d %b %Y %H:%M}**. "
    f"It assessed **{len(assessed_early):,}** later intervals. "
    "These are observations for human review, not confirmed faults. "
    "The original Notebook 02 evaluation is preserved."
)


**Separate historical machine assessment**

System,Pre-test valid readings,Scored test intervals,Review observations,Models trained
TEC_48S,"6,353",1503.0000,0,lower_load


> **💡 What this tells us**
>
> The machine detector learned only from readings before **24 Mar 2024 08:00**. It assessed **1,503** later intervals. These are observations for human review, not confirmed faults. The original Notebook 02 evaluation is preserved.

### 🏁 8. Final answers — what have we actually learned?
The answers below come from the **current notebook run**, not values copied from earlier charts. They summarize the review workload, reliability of the electrical-condition comparison and the handoff to Notebook 03.


In [11]:

# evidence and limitations.
machine_alerts = int(monitors["machine"].review_flag.sum())
pv_alerts = int(monitors["pv"].review_flag.sum())
calib = 100 * REVIEW_SHARE

machine_assessed = review_table.loc[
    review_table.Dataset == "TEC_48S"
].iloc[0]

pv_assessed = review_table.loc[
    review_table.Dataset == "IPE_PV"
].iloc[0]

show_cards([
    (
        "Machine reviews",
        f"{machine_alerts:,}",
        f"{machine_assessed.Review_pct:.2f}% of assessed observations"
    ),
    (
        "Solar reviews",
        f"{pv_alerts:,}",
        f"{pv_assessed.Review_pct:.2f}% of assessed observations"
    ),
    (
        "Review episodes",
        f"{len(events):,}",
        "Grouped alerts · not fault events"
    ),
    (
        "Calibration budget",
        f"{calib:.0f}%",
        "Review workload setting, not a failure rate"
    ),
    (
        "Degradation validation",
        "Not validated",
        "No verified degradation or maintenance labels"
    ),
    (
        "MILP handoff",
        "Time-safe",
        "Use available_at, not interval start"
    ),
])

report(
    "**1. Are the measurements reliable?** "
    "Not all. Quality checks distinguish usable readings "
    "from missing or incomplete measurements. The original "
    "solar AC-power source ends on April 8, 2024; later "
    "missing measurements are not equipment faults.\n\n"

    f"**2. Did we find unusual behavior?** "
    f"Yes: **{machine_alerts:,} machine** and "
    f"**{pv_alerts:,} solar** review intervals, grouped into "
    f"**{len(events):,} episodes**. These are investigation "
    "leads, not confirmed failures.\n\n"

    "**3. Do forecast errors confirm faults?** "
    "No. Forecast errors describe unexpected measurements, "
    "not verified equipment failures.\n\n"

    f"**4. Is the machine degrading?** "
    f"**Not established.** {condition_result}. "
    "**Actual degradation detection remains unvalidated.** "
    "The analysis compares electrical behavior under similar "
    "measured power, not verified identical machining conditions. "
    "Sensor inconsistencies, missing maintenance records and "
    "unavailable degradation labels prevent independent validation. "
    "No degradation rate or remaining useful life is claimed.\n\n"

    "**5. Is the output suitable for Notebook 03?** "
    "Yes, as an advisory input. The available_at timestamp "
    "ensures decisions use only information already available. "
    "Review flags are not safety commands or confirmed faults."
)


> **💡 What this tells us**
>
> **1. Are the measurements reliable?** Not all. Quality checks distinguish usable readings from missing or incomplete measurements. The original solar AC-power source ends on April 8, 2024; later missing measurements are not equipment faults.
> 
> **2. Did we find unusual behavior?** Yes: **44 machine** and **53 solar** review intervals, grouped into **56 episodes**. These are investigation leads, not confirmed failures.
> 
> **3. Do forecast errors confirm faults?** No. Forecast errors describe unexpected measurements, not verified equipment failures.
> 
> **4. Is the machine degrading?** **Not established.** No clear persistent P/S difference (weekly shift -0.004; 6 early / 4 later weeks) at 0.067–0.069 kW (67–69 W); overall machine wear not assessed. **Actual degradation detection remains unvalidated.** The analysis compares electrical behavior under similar measured power, not verified identical machining conditions. Sensor inconsistencies, missing maintenance records and unavailable degradation labels prevent independent validation. No degradation rate or remaining useful life is claimed.
> 
> **5. Is the output suitable for Notebook 03?** Yes, as an advisory input. The available_at timestamp ensures decisions use only information already available. Review flags are not safety commands or confirmed faults.